<a href="https://colab.research.google.com/github/HarshalKushwaha0027/FlyRankAI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshalKushwaha0027/FlyRankAI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


The Rule: A page is flagged for a "Quick Win CTR Fix" if it ranks on the first page of Google (average position <= 10) but suffers from an abnormally low CTR (< 2%). The baseline_score is set to the page's total impressions, meaning we rank the highest-traffic "broken" pages at the very top of the queue.

Reason Codes & Actions:

    Reason Code: page1_low_ctr -> Action: update_metadata_and_title

    Reason Code: ok_performance -> Action: monitor (Score = 0)

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata

# Re-establish the connection
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Build the queue using DuckDB SQL to encode the rule
query_queue = f"""
    WITH page_stats AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) as impressions,
            SUM(gsc_clicks) as clicks,
            SUM(gsc_sum_position) as sum_position
        FROM read_parquet('{table_path}')
        WHERE month = '2026-03'
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) > 100
    ),
    calculated_metrics AS (
        SELECT
            content_hash_id,
            client_hash_id,
            impressions,
            clicks * 1.0 / impressions as ctr,
            sum_position * 1.0 / impressions as avg_pos
        FROM page_stats
    )
    SELECT
        content_hash_id,
        client_hash_id,
        impressions,
        ROUND(ctr, 4) as ctr,
        ROUND(avg_pos, 1) as avg_pos,
        -- The Rule: Score equals impressions IF it's on page 1 with terrible CTR, else 0
        CASE
            WHEN avg_pos <= 10 AND ctr < 0.02 THEN impressions
            ELSE 0
        END as baseline_score,
        -- ONE Reason Code
        CASE
            WHEN avg_pos <= 10 AND ctr < 0.02 THEN 'page1_low_ctr'
            ELSE 'ok_performance'
        END as reason_code,
        -- ONE Action Label
        CASE
            WHEN avg_pos <= 10 AND ctr < 0.02 THEN 'update_metadata_and_title'
            ELSE 'monitor'
        END as action_label
    FROM calculated_metrics
    ORDER BY baseline_score DESC
"""

df_queue = con.sql(query_queue).df()

# Ensure the output directory exists
os.makedirs('work/outputs', exist_ok=True)

# Write the CSV out of git as required by the CI leak-guard
csv_path = 'work/outputs/baseline_action_score.csv'
df_queue.to_csv(csv_path, index=False)

print(f"Ranked queue built and saved to {csv_path}")
print(f"Top 10 rows flagged for action:")
display(df_queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue built and saved to work/outputs/baseline_action_score.csv
Top 10 rows flagged for action:


,content_hash_id,client_hash_id,impressions,ctr,avg_pos,baseline_score,reason_code,action_label
0,content_eadb33b5df496f4a,client_e547b89c05043229,617124.0,0.0092,2.3,617124.0,page1_low_ctr,update_metadata_and_title
1,content_ec2e0346994fb5a5,client_e547b89c05043229,245276.0,0.0060,2.8,245276.0,page1_low_ctr,update_metadata_and_title
2,content_0e03de7680314cd5,client_e547b89c05043229,221310.0,0.0033,2.5,221310.0,page1_low_ctr,update_metadata_and_title
3,content_44f34c0a90047651,client_23a62021009f63c4,212404.0,0.0001,0.7,212404.0,page1_low_ctr,update_metadata_and_title
4,content_7172a7fad43f0998,client_62f4a7e64f5e0096,205867.0,0.0042,3.3,205867.0,page1_low_ctr,update_metadata_and_title
5,content_e7b5dd4dff461ad2,client_08a6a72ff48e62c0,205045.0,0.0119,4.5,205045.0,page1_low_ctr,update_metadata_and_title
6,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,0.0014,2.5,203497.0,page1_low_ctr,update_metadata_and_title
7,content_f107e54b10b43725,client_62f4a7e64f5e0096,195997.0,0.0051,3.2,195997.0,page1_low_ctr,update_metadata_and_title
8,content_b99ea6861864dea5,client_62f4a7e64f5e0096,194337.0,0.0019,4.6,194337.0,page1_low_ctr,update_metadata_and_title
9,content_4ffe18112a5642e3,client_e547b89c05043229,186983.0,0.0031,2.4,186983.0,page1_low_ctr,update_metadata_and_title


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

content_eadb33b5df496f4a: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 617124.0, CTR: 0.0092). This recommendation would be wrong if the page is earning impressions primarily from broad navigational queries for a competitor's brand where users naturally click the official domain instead.

content_ec2e0346994fb5a5: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 245276.0, CTR: 0.0060). This recommendation would be wrong if Google is displaying a Featured Snippet or AI Overview for the main query, satisfying user intent directly on the SERP without requiring a click.

content_0e03de7680314cd5: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 221310.0, CTR: 0.0033). This recommendation would be wrong if the content tag or page structure was updated very recently and GSC impression logs reflect old metadata while CTR is still adjusting.

content_44f34c0a90047651: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 212404.0, CTR: 0.0001). This recommendation would be wrong if the near-zero CTR is caused by automated scraping bots or ad crawlers artificially inflating impression counts without genuine human searches.

content_7172a7fad43f0998: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 205867.0, CTR: 0.0042). This recommendation would be wrong if the page targets broad informational intent where the search query intent severely mismatches our product offering.

content_e7b5dd4dff461ad2: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 205045.0, CTR: 0.0119). This recommendation would be wrong if the title is already highly optimized for conversion rate on-site rather than click acquisition from search engines.

content_8d7d99f109e19a2: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 203497.0, CTR: 0.0014). This recommendation would be wrong if the page is scheduled for deprecation or a 301 redirect as part of an upcoming site migration.

content_f107e54b10b43725: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 195997.0, CTR: 0.0051). This recommendation would be wrong if the rank position is hovering artificially on image/news carousels rather than traditional web text results.

content_b99ea6861864dea5: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 194337.0, CTR: 0.0019). This recommendation would be wrong if the page ranks for a high-volume secondary keyword where the primary search intent requires a different content format (e.g., video or calculator).

content_4ffe18112a5642e3: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 186983.0, CTR: 0.0031). This recommendation would be wrong if a recent title change caused a temporary CTR drop that hasn't reached statistical significance yet.

content_acbcc847f8996314: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 170808.0, CTR: 0.0015). This recommendation would be wrong if the high impression count is driven by seasonal traffic spikes that have already passed.

content_471d9cabce329a66: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 164885.0, CTR: 0.0024). This recommendation would be wrong if the page is a B2B landing page targeting ultra-niche buyers where low overall CTR is expected due to selective intent.

content_512dbad65bd5ade9: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 154358.0, CTR: 0.0162). This recommendation would be wrong if the CTR is already at the industry benchmark for this specific high-competition search cluster.

content_987d251ee617d9c6: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 152806.0, CTR: 0.0062). This recommendation would be wrong if the page ranks well due to domain authority but the page content itself is out of stock or inactive.

content_fd2117c2c6790e4b: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 151166.0, CTR: 0.0027). This recommendation would be wrong if the title tag is restricted by legal or compliance guidelines that prevent changing the snippet phrasing.

content_34a70fea29d15f24: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 143019.0, CTR: 0.0003). This recommendation would be wrong if impressions are coming from international locations where our service is unavailable, discouraging clicks.

content_e241d6415ac9e534: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 142304.0, CTR: 0.0024). This recommendation would be wrong if the page serves as a gateway/hub page designed strictly for internal navigation rather than direct organic entry.

content_f43118e089ecc69a: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 139417.0, CTR: 0.0014). This recommendation would be wrong if the keyword exhibits heavy ad presence above organic results, pushing organic clicks down regardless of title optimization.

content_f352b7cfd0b2f434: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 136098.0, CTR: 0.0021). This recommendation would be wrong if the page ranks for long-tail queries where the current meta description is already accurately filtering out non-converting users.

content_8e1334d6356668e3: Action: update_metadata_and_title, Reason: page1_low_ctr. Confidence: High (Impressions: 134984.0, CTR: 0.0000). This recommendation would be wrong if zero clicks stem from a broken canonical tag or indexation issue rather than poor copy.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Print the Top-20 review template

display(df_queue[['content_hash_id', 'action_label', 'reason_code', 'impressions', 'ctr']].head(20))

,content_hash_id,action_label,reason_code,impressions,ctr
0,content_eadb33b5df496f4a,update_metadata_and_title,page1_low_ctr,617124.0,0.0092
1,content_ec2e0346994fb5a5,update_metadata_and_title,page1_low_ctr,245276.0,0.0060
2,content_0e03de7680314cd5,update_metadata_and_title,page1_low_ctr,221310.0,0.0033
3,content_44f34c0a90047651,update_metadata_and_title,page1_low_ctr,212404.0,0.0001
4,content_7172a7fad43f0998,update_metadata_and_title,page1_low_ctr,205867.0,0.0042
5,content_e7b5dd4dff461ad2,update_metadata_and_title,page1_low_ctr,205045.0,0.0119
6,content_8d7d99f109e19aa2,update_metadata_and_title,page1_low_ctr,203497.0,0.0014
7,content_f107e54b10b43725,update_metadata_and_title,page1_low_ctr,195997.0,0.0051
8,content_b99ea6861864dea5,update_metadata_and_title,page1_low_ctr,194337.0,0.0019
9,content_4ffe18112a5642e3,update_metadata_and_title,page1_low_ctr,186983.0,0.0031


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks Analysis:**
The weakest picks generated by this rule are the pages sitting right on the edge of our hard thresholds. Because the rule uses strict cutoffs (`avg_pos <= 10` and `ctr < 0.02`), a page with an average position of 10.1 and a terrible 0.005 CTR gets a score of 0 and is completely ignored. Search performance is continuous, but this heuristic treats it as binary. Furthermore, by ranking strictly by total impressions, we might prioritize a massive page with a borderline 1.9% CTR over a medium-sized page with a truly broken 0.0% CTR.

**Leakage Check:**
I can confirm no data leakage occurred in this baseline.
1. **No future windows:** The data is strictly bounded to the historical month of March 2026.
2. **No product flags:** The query only uses raw, observable metrics (`gsc_impressions`, `gsc_clicks`, `gsc_sum_position`) rather than proprietary labels or future target variables.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. Leakage Check Verification
print("--- Leakage Check: Allowed Columns Only ---")
print("Columns used in queue:", df_queue.columns.tolist())

print("\n--- Leakage Check: Date Boundary ---")
# Verify that no data from April (the future prediction window) leaked into our March features
date_check = con.sql(f"""
    SELECT MIN(report_date) as earliest_date, MAX(report_date) as latest_date
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
""").df()
display(date_check)

latest_date = str(date_check['latest_date'].iloc[0])[:10]
if latest_date <= '2026-03-31':
    print("✅ PASS: No future data leaked. Max date is securely within March.")
else:
    print("❌ FAIL: Future data detected!")

--- Leakage Check: Allowed Columns Only ---
Columns used in queue: ['content_hash_id', 'client_hash_id', 'impressions', 'ctr', 'avg_pos', 'baseline_score', 'reason_code', 'action_label']

--- Leakage Check: Date Boundary ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date
0,2026-03-01,2026-03-31


✅ PASS: No future data leaked. Max date is securely within March.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.